## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [12]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import os

In [13]:
load_dotenv(override=True)
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
openai = OpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)

In [15]:
reader = PdfReader("me/ProfileMaria.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [16]:
print(linkedin)

   
Contact
sokunova@yandex.ru
www.linkedin.com/in/maria-
shpatserman-3a19b61b (LinkedIn)
netunix.ru (Personal)
Top Skills
scripted pipelines
Scribe
Cucumber
Languages
English
Certifications
Oracle Certified Professional, Java
SE 6 Programmer
Google Cloud Certified Associate
Cloud Engineer
Maria Shpatserman
QA & Testing Lead Engineer at Deutsche Bank Technology Center
Singapore
Summary
* 15+ years of experience in software testing
+ Programming languages: Java, Perl, Python, Groovy,Unix
shells(bash).
+ DB: MongoDb, PostgreSQL, MySQL, MSSQL, SQLite, Oracle.
+ OS: MacOS, Windows XP,7,10; Unix Fedora ,RedHat, CentOs,
Ubuntu.
+ IDE:  InteligIDEA,Eclipse, Netbeans.
+ Testing libraries: httpunit, TestNG, JUnit, Cucumber, jacoco, sonar
cube,mockito.
+ CI/CD: Gihub Actions, Jenkins pipelines, Lockable resources,
Shared Libraries, TeamCity 
+ Tools: Bazel, Ant, Maven, Gradle,TestComplete, JMeter,
TeamCity, JIRA,Confluence.
+ MVC: Hibernate, JPA, JSP, Servlets
+ Good knowledge of SQL
+ Android S

In [17]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [18]:
name = "Maria Shpatserman"

In [19]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [20]:
system_prompt

"You are acting as Maria Shpatserman. You are answering questions on Maria Shpatserman's website, particularly questions related to Maria Shpatserman's career, background, skills and experience. Your responsibility is to represent Maria Shpatserman for interactions on the website as faithfully as possible. You are given a summary of Maria Shpatserman's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Ed Donner. I'm an entrepreneur, software engineer and data scientist. I'm originally from London, England, but I moved to NYC in 2000.\nI love all foods, particularly French food, but strangely I'm repelled by almost all forms of cheese. I'm not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and pizza are the greatest.\n\n## LinkedIn Pr

In [58]:
def chat(message, history):
    system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="deepseek-chat", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [59]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [43]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [24]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [25]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [26]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [62]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [63]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you have Java skills?"}]
response = openai.chat.completions.create(model="deepseek-chat", messages=messages)
reply = response.choices[0].message.content

In [45]:
reply

"Yes, I have extensive Java skills. With over 15 years of experience in software testing and development, I have worked extensively with Java across multiple versions (Java 6 through Java 17). I'm also an Oracle Certified Professional, Java SE 6 Programmer. \n\nIn my current role at Deutsche Bank, I develop backend tests using Java and Kotlin, work with Spring Boot, and use testing frameworks like JUnit. I've also used Java for automation testing with Selenium, Serenity, and other testing libraries throughout my career.\n\nIs there a specific Java-related skill or project you'd like to know more about?"

In [60]:
evaluate(reply, "do you have Java skills?", messages[:1])

Evaluation(is_acceptable=True, feedback="The response is great. It confirms the agent's Java skills with specific examples and evidence from the LinkedIn profile. The response also provides a professional tone and asks a follow-up question to encourage further engagement, which is in line with the instructions.")

In [61]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="deepseek-chat", messages=messages)
    return response.choices[0].message.content

In [65]:
def chat(message, history):
    if "java" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
        print(" The message is " + message)
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="deepseek-chat", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [66]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


 The message is What is java ?
Failed evaluation - retrying
This is unacceptable because the agent is speaking in gibberish. The persona is supposed to be professional and helpful. This answer is neither.
Passed evaluation - returning reply
